In [1]:
from langsmith import Client
import retail_multi_agent_and_fast_api  #import our code
from typing_extensions import Annotated, TypedDict
from openai import OpenAI
import json
from dotenv import load_dotenv
import os
import google.generativeai as genai
import asyncio

load_dotenv("./.env")  # Load .env file
client = OpenAI(base_url="http://34.87.50.217:8000/v1", api_key="not-needed")
langSmithClient = Client()

D:\miniconda3\envs\fyp_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Create a dataset

In [5]:
examples = [
    {
        "question": "What are the top 5 products sold in store S001 during the last day of December 2023?",
        "response": """
            **Summary of Results for User Query: "What are the top 5 products sold in store S001 during the last day of December 2023?"**
            The top 5 products sold in store S001 on December 31, 2023, based on units sold and revenue are as follows:

            1. **Product P0011**: 267 units sold, generating **$18,308.46** in revenue.
            2. **Product P0007**: 181 units sold, generating **$12,803.94** in revenue.
            3. **Product P0015**: 164 units sold, generating **$5,914.33** in revenue.
            4. **Product P0004**: 158 units sold, generating **$10,742.74** in revenue.
            5. **Product P0017**: 156 units sold, generating **$7,627.93** in revenue.

            **Key Insights:**
            - The sales data is consistent and shows no significant trends or drops in performance across the products.
            - All products contributed positively to the store’s revenue on the last day of December 2023.
            - There are no outliers or anomalies in the data provided.

            The results have been verified for correctness and do not require any further follow-up.
        """,
    },
    {
        "question": "What should we restock for the store S001 during the end of February 2022?",
        "response": """
            **Restock Plan for Store S001 – End of February 2022**

            Based on the inventory data and average sales trends for the end of February 2022, the following restock recommendation is provided:

            | Product ID | Recommended Restock Quantity | Justification |
            |------------|-----------------------------|----------------|
            | **P0004**  | **82 units**                | Current inventory is 55 units, and average units sold last month were 137 units. Restocking 82 units will help ensure sufficient stock to meet demand through the end of February 2022, based on the current sales trend. |

            **Key Insights:**
            - Product P0004 is the only item identified that requires restocking based on current inventory levels and historical sales data.
            - The restock recommendation for P0004 is calculated to align with the average sales trend, ensuring the store is well-stocked for the end of February.

            No other products require restocking based on the provided data. The restock plan is concise and directly addresses the user's request for restocking recommendations for Store S001.
        """
    },
    {
        "question": "Can you give me a summary of the sales performance for Store S001 in December 2022?",
        "response": """
            Store S001 sales performance for December 2022:
            - **Units Sold**: 81,773 in December 2022, down from 86,119 in November 2022.
            - **Revenue**: $3,981,382.35 in December 2022, down from $4,205,078.03 in November 2022 (a 5.3% decrease).
            - **Top Products**: P0004 and P0014 were top sellers in December, while P0002, P0003, and P0011 led in November.
            - **Sales Trend**: Units sold and revenue both declined month-over-month, indicating a slight slowdown.
            - **Revenue per Unit**: P0002 in November had higher revenue per unit ($500.00) compared to P0014 in December ($522.00).
            - **Anomaly**: P0019 saw a revenue increase of $192,810.90 in December despite slightly lower unit sales.

            OR

            Store S001 had strong sales performance in December 2022.
            - **Total units sold**: 81,773 units.
            - **Total revenue**: $3,981,382.35.
            - **Top categories by units sold**: Clothing (17,457), Toys (16,919), Furniture (16,547), Electronics (16,100), Groceries (14,750).
            - **Top category by revenue**: Clothing ($877,258.92), followed by Toys ($808,667.37) and Furniture ($770,625.64).
            - **Revenue per unit**: Electronics had the highest revenue per unit ($49.32), while Groceries had the lowest ($50.01).
            - **Key observation**: Clothing was the top-performing category in both units and revenue.
        """
    }, 
    {
        "question": "What should we restock for store S002 during December 2023.",
        "response": """
            No restocking is required for store S002 during December 2023. All products have sufficient inventory levels to cover next month's demand, based on the criteria that current inventory should be at least 60% of previous month's demand.

            - All products have inventory levels above 60% of average units sold for December 2023.
            - No products show signs of low inventory or potential stockouts.
            - Inventory levels are well above the threshold for restocking.
            - No follow-up action is needed.
        """
    }, 
    {
        "question": "Which product category has the most sales from store s001 in December 2023",
        "response": """
            The product category with the most sales from store s001 in December 2023 is **Electronics**, with **18,432 units sold** and **$962,646.11 in revenue**.
        """
    }, 
    
    {
        "question": "Which product category has the most sales in December 2023",
        "response": """
            The product category with the most sales in December 2023 is **Electronics**, with **89,754 units sold**.

            - **Top categories by units sold**: Electronics (89,754), Furniture (87,907), Toys (82,523), Groceries (79,569), Clothing (78,919).
            - **Top categories by revenue**: Electronics ($4,428,238.00), Furniture ($4,348,922.59), Toys ($4,103,476.08), Groceries ($3,970,492.55), Clothing ($3,794,381.51).
            - **Revenue per unit**: Electronics had the highest at $49.33, Clothing the lowest at $48.13.
            - **Total revenue**: $20,645,510.74.
            - **Total units sold**: 418,672.
        """
    },
    {
        "question": "What should we restock in the 'Groceries' category for store S003 in December 2023?",
        "response": """
            No restocking is required for the 'Groceries' category in store S003 for December 2023. All products currently have sufficient inventory levels to cover next month's demand, as the inventory levels meet or exceed 60% of the previous month's demand.
        """
    },
    {
        "question": "How did the sales in store S001 compare to store S002 in December 2023?",
        "response": """
            In December 2023, store S002 had higher sales than store S001. S002 sold 85,902 units for a revenue of $4,268,454.70, while S001 sold 84,381 units for a revenue of $4,131,982.93.
        """
    },
    {
        "question": "How did Product P005 perform in both November and December 2023 in Store S001?",
        "response": """
            Requested data is not available.
        """    
    },
    {
        "question": "How did Product P0005 perform in both November and December 2023 in Store S001?",
        "response": """
            Product P0005 sold 3,356 units in November 2023 and 5,097 units in December 2023 in Store S001, generating $199,948.13 and $263,338.34 in revenue respectively.
        """     
    },
    
    #add 5 more test cases
    {
        "question" : "What are the top 5 products sold in store S0001 during the last day of December 2023?",
        "response" : """
            Requested data is not available.
        """
    },
    {
        "question": "What should we restock in the 'Tech Gadget' category for store S003 in December 2023?",
        "response": """
            Requested data is not available.
        """
    },
    {
        "question": "What should we restock for the store S001 during the end of February 2001?",
        "response": """
            Requested data is not available.
        """
    },
    {
        "question": "How did the sales in store S010 compare to store S002 in December 2023?",
        "response": """
            The sales data for store S010 are not available.
        """
    },
    {
        "question": "What should we restock for store S010 during December 2023.",
        "response": """
            Requested data is not available.
        """
    }, 
]
dataset_name = "Retail AI: Final Response-v8"

if not langSmithClient.has_dataset(dataset_name=dataset_name):
    dataset = langSmithClient.create_dataset(dataset_name=dataset_name)
    langSmithClient.create_examples(
        inputs=[{"question": ex["question"]} for ex in examples],
        outputs=[{"response": ex["response"]} for ex in examples],
        dataset_id=dataset.id
    )

### Define application logic to be evaluated

In [6]:
# Target function
async def run_graph(inputs: dict) -> dict:
    """Run graph and track the final response."""
    result = await retail_multi_agent_and_fast_api.multi_agent.ainvoke({"messages": [
        retail_multi_agent_and_fast_api.HumanMessage(content=inputs['question']),
    ]}, config={"env": "test", "recursion_limit": 100})
    return {"response": result["messages"][-1].content}

### Define evaluator

In [4]:
#Qwen3

# LLM-as-judge instructions
grader_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION, the GROUND TRUTH (correct) RESPONSE, and the STUDENT RESPONSE.

Here is the grade criteria to follow:
(1) Grade the student responses based ONLY on their factual accuracy relative to the ground truth answer.
(2) Ensure that the student response does not contain any conflicting statements.
(3) It is OK if the student response contains more information than the ground truth response, as long as it is factually accurate relative to the ground truth response.

Correctness:
True means that the student's response meets all of the criteria.
False means that the student's response does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Please respond in JSON format with the following structure:
{
    "reasoning": "Your step-by-step reasoning here",
    "is_correct": true/false
}"""


# LLM-as-judge output schema (keeping for type hints)
class Grade(TypedDict):
    """Compare the expected and actual answers and grade the actual answer."""
    reasoning: Annotated[str, ..., "Explain your reasoning for whether the actual response is correct or not."]
    is_correct: Annotated[bool, ..., "True if the student response is mostly or exactly correct, otherwise False."]


# Custom judge function using your client
async def grade_with_custom_llm(user_prompt: str) -> Grade:
    """Grade using your custom LLM client."""
    full_prompt = f"{grader_instructions}\n\n{user_prompt}"

    try:
        response = client.chat.completions.create(
            model="Qwen/Qwen3-4B",
            messages=[{"role": "user", "content": full_prompt.strip() + "\n/no_think"}],
            max_tokens=500,  # Increased for detailed reasoning
            temperature=0.1,  # Lower temperature for more consistent grading
        )

        # Extract the response content
        response_content = response.choices[0].message.content.replace("<think>", "").replace("</think>", "").strip()

        # Try to parse JSON response
        try:
            grade_data = json.loads(response_content)
            return {
                "reasoning": grade_data.get("reasoning", ""),
                "is_correct": grade_data.get("is_correct", False)
            }
        except json.JSONDecodeError:
            # Fallback parsing if JSON parsing fails
            # This is a simple fallback - you might want to make this more robust
            reasoning = response_content
            is_correct = "true" in response_content.lower() and "correct" in response_content.lower()

            return {
                "reasoning": reasoning,
                "is_correct": is_correct
            }

    except Exception as e:
        print(f"Error in grading: {e}")
        return {
            "reasoning": f"Error occurred during grading: {str(e)}",
            "is_correct": False
        }


# Evaluator function
async def final_answer_correct(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """Evaluate if the final response is equivalent to reference response."""
    user_prompt = f"""QUESTION: {inputs['question']}
GROUND TRUTH RESPONSE: {reference_outputs['response']}
STUDENT RESPONSE: {outputs['response']}"""

    grade = await grade_with_custom_llm(user_prompt)

    return {
        "key": "final_answer_correct",
        "score": 1.0 if grade["is_correct"] else 0.0,  # Numeric score
        "comment": grade["reasoning"]  # Reasoning goes in comment field
    }

In [7]:
#GEMINI

# Configure Gemini API
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))  # Set your API key as environment variable

# LLM-as-judge instructions (same as before)
grader_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION, the GROUND TRUTH (correct) RESPONSE, and the STUDENT RESPONSE.

Here is the grade criteria to follow:
(1) Grade the student responses based ONLY on their factual accuracy relative to the ground truth answer.
(2) Ensure that the student response does not contain any conflicting statements.
(3) It is OK if the student response contains more information than the ground truth response, as long as it is factually accurate relative to the ground truth response.

Correctness:
True means that the student's response meets all of the criteria.
False means that the student's response does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Please respond in JSON format with the following structure:
{
    "reasoning": "Your step-by-step reasoning here",
    "is_correct": true/false
}"""


# LLM-as-judge output schema (keeping for type hints)
class Grade(TypedDict):
    """Compare the expected and actual answers and grade the actual answer."""
    reasoning: Annotated[str, "Explain your reasoning for whether the actual response is correct or not."]
    is_correct: Annotated[bool, "True if the student response is mostly or exactly correct, otherwise False."]


# Initialize Gemini model
model = genai.GenerativeModel('models/gemini-2.0-flash')


# Custom judge function using Gemini
async def grade_with_gemini(user_prompt: str) -> Grade:
    """Grade using Gemini 2.5 API."""
    full_prompt = f"{grader_instructions}\n\n{user_prompt}"

    try:
        # Configure generation parameters
        generation_config = genai.types.GenerationConfig(
            temperature=0.1,  # Low temperature for consistent grading
            top_p=0.8,
            top_k=40,
            max_output_tokens=500,
            response_mime_type="application/json",  # Force JSON response
        )

        # Generate response
        response = await asyncio.to_thread(
            model.generate_content,
            full_prompt,
            generation_config=generation_config
        )

        # Extract the response content
        response_content = response.text.strip()

        # Try to parse JSON response
        try:
            grade_data = json.loads(response_content)
            return {
                "reasoning": grade_data.get("reasoning", ""),
                "is_correct": grade_data.get("is_correct", False)
            }
        except json.JSONDecodeError as e:
            print(f"JSON parsing failed: {e}")
            print(f"Response content: {response_content}")

            # Fallback parsing if JSON parsing fails
            reasoning = response_content
            is_correct = "true" in response_content.lower() and "correct" in response_content.lower()

            return {
                "reasoning": reasoning,
                "is_correct": is_correct
            }

    except Exception as e:
        print(f"Error in grading with Gemini: {e}")
        return {
            "reasoning": f"Error occurred during grading: {str(e)}",
            "is_correct": False
        }


# Evaluator function
async def final_answer_correct(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """Evaluate if the final response is equivalent to reference response."""
    user_prompt = f"""QUESTION: {inputs['question']}
GROUND TRUTH RESPONSE: {reference_outputs['response']}
STUDENT RESPONSE: {outputs['response']}"""

    grade = await grade_with_gemini(user_prompt)

    return {
        "key": "final_answer_correct",
        "score": 1.0 if grade["is_correct"] else 0.0,  # Numeric score
        "comment": grade["reasoning"]  # Reasoning goes in comment field
    }

### Run evaluation


In [ ]:
# Evaluation job and results
experiment_results = await langSmithClient.aevaluate(
    run_graph,
    data=dataset_name,
    evaluators=[final_answer_correct],
    experiment_prefix="retail-ai",
    num_repetitions=5,
    max_concurrency=3,
)

View the evaluation results for experiment: 'retail-ai-02fb37c3' at:
https://smith.langchain.com/o/72a23be0-1a7c-46bd-b1d9-934d02fccd9f/datasets/e36effa0-36f5-49ed-aa6b-9797fef5a1fb/compare?selectedSessions=3b55dcff-fe19-49d3-9284-81787636096a




0it [00:00, ?it/s]

🤖 Agent responding: 🤖 Agent responding: 🤖 Agent responding: <think><think><think>





</think></think></think>







In [6]:
experiment_results.to_pandas()

,inputs.question,outputs.response,error,reference.response,feedback.final_answer_correct,execution_time,example_id,id
0,Which product category has the most sales in D...,The product category with the most sales in De...,None,\n The product category with the mo...,1.0,34.233107,42dcbac8-5981-493b-9593-4b1dc1de8f3f,929579e2-1d51-4aad-b9b0-9d250c698e04
1,Which product category has the most sales from...,The product category with the most sales from ...,None,\n The product category with the mo...,1.0,36.983454,127d80dc-028e-4cc4-b116-61a1b0e1d84b,20a1fd04-ef2a-490f-98b3-59bc14a4e4f5
2,What are the top 5 products sold in store S001...,The top 5 products sold in store S001 during t...,None,\n **Summary of Results for User Qu...,1.0,70.344797,5230496b-ddbd-49a3-9fb3-25e34ed01cb3,6ee73872-67c4-4e28-967c-57ba5d7ac30d
3,What should we restock for store S002 during D...,No restocking is required for store S002 durin...,None,\n No restocking is required for st...,1.0,53.876947,b7237c77-e01e-4027-9830-16573226e389,fd54521d-1bf8-4e30-8bbd-ba3ce0be3279
4,What should we restock for the store S001 duri...,Based on the inventory data for store S001 dur...,None,\n **Restock Plan for Store S001 – ...,1.0,61.994842,a5f08a2b-f053-422d-8257-1b4c739699d4,d3c0302d-dab2-4f33-b35f-3b0e69f0a086
5,Which product category has the most sales in D...,The product category with the most sales in De...,None,\n The product category with the mo...,1.0,31.281421,42dcbac8-5981-493b-9593-4b1dc1de8f3f,fb57b190-7517-454c-bde7-2cd045d5bbae
6,Which product category has the most sales from...,The product category with the most sales (by u...,None,\n The product category with the mo...,0.0,54.198748,127d80dc-028e-4cc4-b116-61a1b0e1d84b,7a735126-c023-4657-9643-5aed1d6d0b56
7,Can you give me a summary of the sales perform...,"Store S001 reported a total revenue of $3,981,...",None,\n Store S001 sales performance for...,1.0,98.811566,d53dd726-1dea-4d26-bd88-bdf13952ecd8,be0f8b13-7559-4495-9cfb-373ccbdc8c46
8,What are the top 5 products sold in store S001...,The top 5 products sold in store S001 during t...,None,\n **Summary of Results for User Qu...,1.0,64.474501,5230496b-ddbd-49a3-9fb3-25e34ed01cb3,7bcbc166-e07e-4146-b195-c5ad899258e4
9,What should we restock for the store S001 duri...,Based on the inventory data for store S001 dur...,None,\n **Restock Plan for Store S001 – ...,1.0,59.533355,a5f08a2b-f053-422d-8257-1b4c739699d4,8ea37487-fffd-4398-b835-bf646b6cc290
